# Exploring the final names dataset

Goal: explore `data/processed/names.txt` — one name per line, the plain names-only
output of the pipeline (consolidation + Phase 2 normalization). This notebook is
read-only: it never modifies anything under `data/`.

Its findings feed the Phase 3 model choices: alphabet size, name-length distribution,
start/end tokens and n-gram sparsity (bigram vs trigram vs MLP sizing).

All charts are text histograms on purpose — the project has no plotting dependency.


In [ ]:
import random
import re
import unicodedata
from collections import Counter
from pathlib import Path

path = Path("../data/processed/names.txt")  # kernel cwd = notebooks/
names = [line for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

len(names), names[:5]

## Volume and uniqueness


In [ ]:
print(f"{len(names)} names loaded")
print(f"{len(set(names))} unique names")
dupes = [name for name, count in Counter(names).items() if count > 1]
print(f"{len(dupes)} duplicated names: {dupes[:10]}")

## Name length

Character-count distribution, plus the longest outliers (often role lists or titles
rather than a single name).


In [ ]:
def ascii_hist(pairs, width=40):
    # Text histogram from (label, count) pairs — no plotting dependency.
    peak = max(count for _, count in pairs)
    for label, count in pairs:
        if isinstance(label, str) and len(label) == 1:
            label = repr(label)  # make spaces visible
        bar = "#" * round(width * count / peak)
        print(f"{label:>6} {bar} {count}")


lengths = sorted(len(name) for name in names)
median = lengths[len(lengths) // 2]
print(
    f"min {lengths[0]}, median {median}, "
    f"mean {sum(lengths) / len(lengths):.2f}, max {lengths[-1]}"
)
print()
ascii_hist(sorted(Counter(lengths).items()))

LONG = 40
long_names = sorted((n for n in names if len(n) > LONG), key=len, reverse=True)
print(f"\n{len(long_names)} names longer than {LONG} characters:")
for name in long_names[:20]:
    print(f"{len(name):>3}  {name}")

## Alphabet

Which characters actually occur, and how often. This decides the model vocabulary:
keep the accents? lowercase everything? drop digits and symbols?


In [ ]:
charset = Counter("".join(names))
print(f"{len(charset)} distinct characters over {sum(charset.values())} total\n")


def char_group(ch):
    if ch == " ":
        return "space"
    if ch.isascii() and ch.isalpha():
        return "ascii letter"
    if ch.isalpha():
        return "non-ascii letter (accented or other script)"
    if ch.isdigit():
        return "digit"
    return "punctuation / symbol"


groups = {}
for ch, freq in charset.items():
    groups.setdefault(char_group(ch), []).append((ch, freq))
group_order = sorted(groups.items(), key=lambda kv: -sum(f for _, f in kv[1]))
for group, items in group_order:
    chars = sorted(ch for ch, _ in items)
    occurrences = sum(freq for _, freq in items)
    print(f"{group:<40} {len(chars):>3} distinct, {occurrences:>6} occurrences")
    print(f"  {' '.join(repr(c) for c in chars)}\n")

print("most common characters:")
ascii_hist(charset.most_common(30))

## First and last letters

With a start/end token, generation starts from the first-letter distribution and stops
on the last-letter one — both matter for plausible names.


In [ ]:
first = Counter(name[0] for name in names)
last = Counter(name[-1] for name in names)
print("first letters:")
ascii_hist(first.most_common(20))
print("\nlast letters:")
ascii_hist(last.most_common(20))

## N-gram sparsity

How many distinct n-grams the dataset actually contains versus how many are possible,
raw and lowercased. This sizes the model: a bigram table is dense, a trigram table is
sparse, an MLP smooths in between.


In [ ]:
def ngram_counts(pool, n, lower=False):
    counts = Counter()
    for name in pool:
        s = name.lower() if lower else name
        padded = f".{s}."  # one start token and one end token
        for i in range(len(padded) - n + 1):
            counts[padded[i : i + n]] += 1
    return counts


alphabet_size = len(set("".join(names)))
for n in (2, 3):
    raw = ngram_counts(names, n)
    low = ngram_counts(names, n, lower=True)
    possible = (alphabet_size + 1) ** n  # rough upper bound on the padded alphabet
    share = len(raw) / possible
    print(
        f"{n}-grams: {len(raw)} distinct raw ({share:.1%} of {possible}), "
        f"{len(low)} lowercased"
    )
    print(f"  top raw:   {raw.most_common(8)}")
    print(f"  top lower: {low.most_common(8)}\n")

## Free exploration

Re-runnable cells: each run draws a new sample, or applies the pattern below.


In [ ]:
N = 30  # sample size; re-run the cell for a new sample
for name in random.sample(names, min(N, len(names))):
    print(name)

In [ ]:
PATTERN = r"^Saint"  # regex applied to names
LIMIT = 80

rx = re.compile(PATTERN, re.IGNORECASE)
hits = [name for name in names if rx.search(name)]
print(f"{len(hits)} names matching /{PATTERN}/")
for name in hits[:LIMIT]:
    print(name)

## Potential anomalies

Control views, judged by eye — nothing is excluded here and the notebook never modifies
`data/`. To trace an anomaly back to its origin, the provenance (source, author) is
still in `data/interim/names.jsonl` and the per-source files: grep the name there.


In [ ]:
# Digits or symbols inside names (not letters, spaces, apostrophes or hyphens)
def has_digit_or_symbol(name):
    return any(ch.isdigit() or (not ch.isalpha() and ch not in " '-’") for ch in name)


odd = [name for name in names if has_digit_or_symbol(name)]
print(f"{len(odd)} names contain a digit or a symbol:")
for name in odd[:40]:
    print(name)

In [ ]:
# Names containing letters from a non-Latin script (Cyrillic, Greek, ...)
def is_latin_letter(ch):
    try:
        return unicodedata.name(ch).startswith("LATIN")
    except ValueError:
        return False


non_latin = [
    name
    for name in names
    if any(ch.isalpha() and not is_latin_letter(ch) for ch in name)
]
print(f"{len(non_latin)} names contain a non-Latin letter:")
for name in non_latin[:40]:
    print(name)

In [ ]:
# Multi-word names: the space becomes a character of the model alphabet
multi = [name for name in names if " " in name]
print(f"{len(multi)} names contain a space; sample of 25:")
for name in random.sample(multi, min(25, len(multi))):
    print(name)

In [ ]:
# Names not starting with an uppercase letter: case-handling anomalies
lower_start = [name for name in names if name[0].islower()]
print(f"{len(lower_start)} names start with a lowercase letter:")
for name in lower_start:
    print(name)